# CaptionBERT structured rotary — lane 1

Use **G4 / RTX PRO6000** via the arrow right of Connect → Change runtime type.
Enable **HF_TOKEN** notebook access. Run setup, the full-batch gate, TensorBoard,
then training. This tagged **v0.3.0** notebook preserves Colab's host packages.

The [registered protocol](https://github.com/AbstractEyes/captionbert-trainer/blob/v0.3.0/docs/structured_campaign_protocol.md)
defines the eight options, paired seeds, repeat, development-only selection and
4M-caption promotion. Each lane runs six models. Logs, safetensors checkpoints,
optimizer/RNG state and TensorBoard events ship to
[the model repo](https://huggingface.co/AbstractPhil/captionbert-8192-rotary).

After an interrupted runtime, repeat setup/gate/TensorBoard/training with the
same tag and lane. Compatible remote checkpoints resume; completed runs are verified
and skipped. The final model format is `model.safetensors` with AutoModel support.
No 8K semantic-quality claim follows from the position limit.


In [ ]:
import os
import pathlib
import subprocess
import sys
import urllib.request

CONFIG = {"release": "v0.3.0", "lane": 0, "env": "/content/captionbert-structured-env"}
required = {"release", "lane", "env"}
if required - CONFIG.keys():
    raise ValueError("Incomplete setup configuration; restore this tagged notebook and rerun setup")
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Choose Colab Python3.11-3.13, then rerun this setup cell")
os.environ["HF_HOME"] = "/content/captionbert-hf"
os.environ["HF_HUB_CACHE"] = "/content/captionbert-hf/hub"
env_dir = pathlib.Path(CONFIG["env"])
python = str(env_dir / "bin/python")
if not pathlib.Path(python).exists():
    bootstrap = pathlib.Path("/content/captionbert-bootstrap")
    subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(bootstrap), "virtualenv>=20.38.0"], check=True)
    bootstrap_env = os.environ.copy()
    bootstrap_env["PYTHONPATH"] = str(bootstrap)
    subprocess.run([sys.executable, "-m", "virtualenv", "--no-download", str(env_dir)], env=bootstrap_env, check=True)
req_dir = pathlib.Path("/content/captionbert-structured-install")
req_dir.mkdir(exist_ok=True)
for name in ("requirements.txt", "requirements-cuda.txt", "requirements-colab.txt"):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/AbstractEyes/captionbert-trainer/{CONFIG['release']}/{name}", req_dir / name)
subprocess.run([python, "-m", "pip", "install", "-r", str(req_dir / "requirements-colab.txt")], check=True)
subprocess.run([python, "-m", "pip", "install", "--no-deps", f"git+https://github.com/AbstractEyes/captionbert-trainer.git@{CONFIG['release']}"], check=True)
subprocess.run([python, "-m", "pip", "check"], check=True)
subprocess.run([python, "-c", "import torch,numpy,scipy,transformers,datasets,safetensors,captionbert_rotary; from torch.utils.tensorboard import SummaryWriter; from captionbert_rotary.modeling_captionbert import CaptionBertModel; print('READY',captionbert_rotary.__version__,torch.__version__,torch.version.cuda); assert torch.cuda.is_available(), 'Select G4 via the arrow beside Connect -> Change runtime type'; print(torch.cuda.get_device_name(0)); assert '6000' in torch.cuda.get_device_name(0), 'This campaign requires the selected G4 / RTX PRO6000'; print('GPU GB',torch.cuda.get_device_properties(0).total_memory/1e9)"], check=True)
from google.colab import userdata
try:
    token = userdata.get("HF_TOKEN")
except Exception as error:
    raise RuntimeError("Enable HF_TOKEN notebook access in the Secrets sidebar, then rerun setup") from error
os.environ["HF_TOKEN"] = token
subprocess.run([python, "-c", "from huggingface_hub import HfApi; print('Hub access:',HfApi().whoami()['name'])"], check=True)
print("Setup ready; next run the full-batch engineering gate, TensorBoard, then this lane's training cell.", flush=True)


In [ ]:
import os
import pathlib
python = "/content/captionbert-structured-env/bin/python"
root = pathlib.Path("/content/captionbert-structured/screen")
root.mkdir(parents=True, exist_ok=True)
if not pathlib.Path(python).exists():
    raise RuntimeError("Run setup cell first; the isolated Python environment is missing")
"""Parent-process output relay for notebook cells and an on-disk console log."""


def run_logged(command, log_path, env=None):
    import os
    from pathlib import Path
    import subprocess
    import time

    child_env = os.environ.copy() if env is None else dict(env)
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONIOENCODING"] = "utf-8"
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Streaming child output; console log: {path}", flush=True)
    started = time.monotonic()
    with path.open("a", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1,
                                   env=child_env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGINT)
            raise
        finally:
            process.stdout.close()
        print(f"Child exited with code {returncode}; elapsed {time.monotonic() - started:.1f}s; "
              f"console log: {path}", flush=True)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    return returncode

run_logged([python, "-m", "captionbert_rotary.structured_preflight", "--full", "--output", str(root / "preflight.json")], root / "preflight-console.log")
print("Full-width batch2048 x256 gate passed. Next start TensorBoard, then training.", flush=True)


In [ ]:
import os
import pathlib
import subprocess
from google.colab import output

root = pathlib.Path("/content/captionbert-structured")
root.mkdir(parents=True, exist_ok=True)
pid_path = root / "tensorboard.pid"
alive = False
if pid_path.exists():
    try:
        os.kill(int(pid_path.read_text()), 0)
        alive = True
    except (OSError, ValueError):
        pass
if not alive:
    log = (root / "tensorboard-console.log").open("a")
    process = subprocess.Popen(["/content/captionbert-structured-env/bin/python", "-m", "tensorboard.main", "--logdir", str(root), "--host", "127.0.0.1", "--port", "6006"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    pid_path.write_text(str(process.pid))
print("TensorBoard: each arm/seed has its own run. Scalar data appear as training starts.")
output.serve_kernel_port_as_iframe(6006, height=420)


In [ ]:
import os
import pathlib
from google.colab import userdata

RUN = {"stage": "screen", "lane": 0, "data": "/content/captionbert-data", "output": "/content/captionbert-structured/screen"}
required = {"stage", "lane", "data", "output"}
if required - RUN.keys():
    raise ValueError("Incomplete run configuration; restore this tagged cell")
python = "/content/captionbert-structured-env/bin/python"
if not pathlib.Path(python).exists():
    raise RuntimeError("Run setup, then the engineering gate, before training")
gate = pathlib.Path(RUN["output"]) / "preflight.json"
if not gate.exists():
    raise RuntimeError("Run the full-batch engineering gate cell before training")
import json
if not json.loads(gate.read_text()).get("passed"):
    raise RuntimeError("Engineering gate failed; inspect its output before starting training")
env = os.environ.copy()
env["HF_TOKEN"] = userdata.get("HF_TOKEN")
env["HF_HOME"] = "/content/captionbert-hf"
env["HF_HUB_CACHE"] = "/content/captionbert-hf/hub"
env["OMP_NUM_THREADS"] = "2"
env["MKL_NUM_THREADS"] = "2"
"""Parent-process output relay for notebook cells and an on-disk console log."""


def run_logged(command, log_path, env=None):
    import os
    from pathlib import Path
    import subprocess
    import time

    child_env = os.environ.copy() if env is None else dict(env)
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONIOENCODING"] = "utf-8"
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Streaming child output; console log: {path}", flush=True)
    started = time.monotonic()
    with path.open("a", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1,
                                   env=child_env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGINT)
            raise
        finally:
            process.stdout.close()
        print(f"Child exited with code {returncode}; elapsed {time.monotonic() - started:.1f}s; "
              f"console log: {path}", flush=True)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    return returncode

print("Starting lane", RUN["lane"] + 1, "with two seeds per registered option. Check the step display and TensorBoard.")
run_logged([python, "-m", "captionbert_rotary.structured_campaign", "--stage", RUN["stage"], "--lane", str(RUN["lane"]), "--data", RUN["data"], "--output", RUN["output"]], pathlib.Path(RUN["output"]) / "console.log", env=env)


In [ ]:
import pathlib
import subprocess
python = "/content/captionbert-structured-env/bin/python"
code = """import json,pathlib
root=pathlib.Path('/content/captionbert-structured/screen')
for directory in sorted(p for p in root.iterdir() if p.is_dir() and '-seed' in p.name):
    result=directory/'result.json'
    if result.exists():
        value=json.loads(result.read_text()); print(directory.name, 'COMPLETE', value['steps'], 'devSTS', value['development']['spearman'], 'cos', value['evaluation']['heldout']['cos_to_target'])
    elif (directory/'train.jsonl').exists():
        last=json.loads((directory/'train.jsonl').read_text().splitlines()[-1]); print(directory.name, 'step',last['step'],'loss',last['loss'],'lr',last['lr'])
print('TensorBoard directories:',len(list(root.glob('*/tensorboard'))))
"""
subprocess.run([python, "-c", code], check=True)
